In [1]:
import os
from pathlib import Path

# Document folder

In [ ]:
doc_dir = "doc_folder_2"

files = list(Path(doc_dir).glob("*")) 
print(files)

[PosixPath('doc_folder_1/Public003.pdf'), PosixPath('doc_folder_1/Public004.pdf')]


In [3]:
file = files[0]
print(file)

doc_folder_1/Public003.pdf


# Basic usage

Using `DocumentConverter` to process a document.

In [ ]:
# from docling.document_converter import DocumentConverter

In [5]:
source = file
converter = DocumentConverter()
doc = converter.convert(source).document

2025-10-17 16:54:15,783 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-17 16:54:15,821 - INFO - Going to convert document batch...
2025-10-17 16:54:15,821 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e647edf348883bed75367b22fbe60347
2025-10-17 16:54:16,050 - WARNING - The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
2025-10-17 16:54:16,050 - INFO - Loading plugin 'docling_defaults'
2025-10-17 16:54:16,052 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-10-17 16:54:16,060 - WARNING - The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
2025-10-17 16:54:16,060 - INFO - Loading plugin 'docling_defaults'
2025-10-17 16:54:16,063 - INFO - Registered ocr engines: ['easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-10-17 16:54:16,554 - INFO - Accelerator device: 'mps'
2025-10-17 16:54:18,918 - I

Serialize the converted result with `export_to_...` command

In [ ]:
# print(doc.export_to_markdown()[:500]) 

<!-- image -->

## VIETTEL AI RACE

## THÍCH Ứ NG MI Ề N TRONG D Ị CH MÁY NƠ RON CHO NGÔN NGỮ ANH - VI Ệ T

TD003

L ầ n ban hành: 1

D ị ch máy là m ộ t trong nh ững hướ ng nghiên c ứ u quan tr ọ ng trong x ử lý ngôn ng ữ t ự nhiên. Trong nh ững năm gần đây, dịch máy nơ ron đã và đang đượ c nghiên c ứ u ph ổ bi ến hơn trong cộng đồ ng d ị ch máy vì hi ệ n t ạ i nó cho ch ất lượ ng d ị ch t ố t hơn so với phương pháp dị ch máy th ố ng kê truy ề n th ố ng. Tuy nhiên, d ịch máy nơ ron l ạ i c ần l


# Annotate images with ollama

In [7]:
import sys
from pathlib import Path

# Thêm đường dẫn project vào sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    PictureDescriptionApiOptions
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc.document import ImageRefMode
from src.config import config

In [8]:
def convert_with_image_annotation(input_doc_path):
    # Đọc cấu hình từ config.yaml
    base_url = config.get("model", "url", default="http://localhost:11434/v1")
    vision_model = config.get("model", "vision_model", default="")
    text_model = config.get("model", "text_generation", default="")
    picture_prompt = config.get("document", "picture_description", "prompt_picture_description", 
                              default="Miêu tả chi tiết hình ảnh sau bằng một đoạn văn.")
    pd_enabled = config.get("document", "picture_description", "enabled", default=True)
    image_scale = config.get("document", "image_resolution_scale", default=2)
    
    # Nếu tắt picture description, không dùng remote API
    if not pd_enabled:
        pipeline_options = PdfPipelineOptions(
            images_scale=image_scale,
            generate_picture_images=False,
            do_picture_description=False,
            picture_description_options=None,
            enable_remote_services=False,
        )
    else:
        # Dùng vision_model nếu có, fallback sang text_model
        model_name = vision_model if vision_model else text_model
        
        # Chuẩn hóa URL cho Ollama OpenAI-compatible API
        if not base_url.endswith("/chat/completions") and not base_url.endswith("/v1/chat/completions"):
            if base_url.endswith("/v1"):
                api_url = f"{base_url}/chat/completions"
            else:
                api_url = f"{base_url}/v1/chat/completions"
        else:
            api_url = base_url
        
        picture_desc_api_option = PictureDescriptionApiOptions(
            url=api_url,
            prompt=picture_prompt,
            params={"model": model_name},
            timeout=60,
        )

        pipeline_options = PdfPipelineOptions(
            images_scale=image_scale,
            generate_picture_images=True,
            do_picture_description=True,
            picture_description_options=picture_desc_api_option,
            enable_remote_services=True,
        )

    converter = DocumentConverter(
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
    )
    conv_res = converter.convert(source=input_doc_path)
    return conv_res

In [9]:
result = convert_with_image_annotation(file)

2025-10-17 16:54:30,127 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-17 16:54:30,132 - INFO - Going to convert document batch...
2025-10-17 16:54:30,133 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 68e059f3e75bb40d4b4d82a1df76f3d7
2025-10-17 16:54:30,133 - INFO - Accelerator device: 'mps'


KeyboardInterrupt: 

In [ ]:
print(result.document.export_to_markdown(mark_annotations = True, include_annotations=True))

<!-- image -->

## 1. L ờ i m ở đầ u

Bài toán nhận diện biển số xe Việt Nam là một bài toán không còn mới, đã được phát triển dựa trên các phương pháp xử lý ảnh truyền thống và cả những kỹ thuật mới sử dụng Deep Learning. Trong bài toán này tôi chỉ phát triển bài toán phát hiện biển số (một phần trong bài toán nhận diện biển số) dựa trên thuật toán YOLO -Tinyv4 với mục đích:

- Hướng dẫn chuẩn bị dữ liệu cho bài toán Object Detection.
- Hướng dẫn huấn luyện YOLO -TinyV4 dùng darknet trên Google Colab.

## 2. Chu ẩ n b ị d ữ li ệ u

## 2.1 Đánh giá bộ d ữ li ệ u

Trong bài viết tôi sử dụng bộ dữ liệu biển số xe máy Việt Nam chứa 1750 ảnh, bạn đọc có thể tải tại đây .

Hình 14.1: Ảnh biển số trong bộ dữ liệu

<!--<annotation kind="description">-->Bái có cảm ảng cấp trong mạnh trị của một vị trí chuyển lại phúc vào thành công tất cả.<!--<annotation/>-->

<!-- image -->

Ảnh biển số xe được trong bộ dữ liệu được chụp từ một camera tại vị trí kiểm soát xe ra vào trong hầm. Do vậy:

- Kích 

# Save to MD with external referenced images

In [ ]:
def export_function_md_with_image_ref(conv_res, output_path:str, replace_blank:str="_"):
    
    output_dir = Path(output_path)
    output_dir.mkdir(parents=True, exist_ok=True)
    doc_filename = conv_res.input.file.stem.replace(" ", replace_blank)

    # Save markdown with externally referenced pictures
    md_filename = output_dir / f"{doc_filename}-with-image-refs.md"
    conv_res.document.save_as_markdown(md_filename, image_mode=ImageRefMode.REFERENCED, include_annotations=True)

In [ ]:
export_function_md_with_image_ref(result, "outputs")